# Phase 5.4: Safety screening of the discovery set

The 412 calibrated discoveries from phase 5.3 are passed through two safety gates
before any of them reach docking. Toxicity is judged by ToxinPred2 and
allergenicity by AlgPred 2.0, each keeping its continuous score alongside the
call, since both tools sit close to their thresholds on short food peptides and
milk, egg and soy proteins are themselves major allergens.

This is a notebook rather than a script because AlgPred 2.0 was run through its
web server, so the pipeline stops halfway and waits for a file to come back.
Section 4 is that pause.

    INPUTS   screening/screening_discovery.csv (412 discoveries, from phase 5.3)
    OUTPUTS  screening/discovery_top412.fasta
             screening/toxinpred2_discovery.csv
             screening/screening_safe_shortlist.csv
             screening/safe_shortlist.fasta
             screening/screening_final_leads.csv

    REQUIREMENTS  pip install pandas toxinpred2

In [1]:
import os

import pandas as pd

# a notebook kernel starts in the notebook's own folder rather than the repo root,
# so step up until the paths below resolve. everything is then repo-root relative,
# matching the scripts
while not os.path.isdir("screening") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

DISCOVERY = "screening/screening_discovery.csv" # 412 calibrated discoveries
TOXPRED = "screening/toxinpred2_discovery.csv" # written by section 2
ALGPRED = "screening/algpred2_safe.csv" # downloaded by hand in section 4

SHORTLIST = "screening/screening_safe_shortlist.csv"
SHORTLIST_FASTA = "screening/safe_shortlist.fasta"
FINAL = "screening/screening_final_leads.csv"

TOX_THRESHOLD = 0.6 # ToxinPred2's own cut, below this is non-toxic
ALG_LABEL = "Non-Allergen" # AlgPred's call at its 0.3 threshold

# keep_default_na=False, or the dipeptide NA would parse as a missing value
disc = pd.read_csv(DISCOVERY, keep_default_na=False)
disc.head()

,peptide_id,sequence,length,sources,enzymes,esm_prob,xgb_prob,consensus,discovery,high_confidence
0,CAND2163,GPFPSIL,7,soybean_beta_conglycinin,gi_cascade;tryp+chymo,0.9976,0.9977,0.9976,True,True
1,CAND2716,FVAPFPEVF,9,bovine_alpha_s1_casein,chymotrypsin;tryp+chymo,0.9985,0.9950,0.9968,True,True
2,CAND1349,FYPEL,5,bovine_alpha_s1_casein,chymotrypsin;tryp+chymo,0.9979,0.9946,0.9962,True,True
3,CAND0979,LPGF,4,chicken_ovalbumin,gi_cascade;tryp+chymo,0.9981,0.9930,0.9956,True,True
4,CAND3121,TSVLMAMGITDVF,13,chicken_ovalbumin,chymotrypsin;tryp+chymo,0.9986,0.9923,0.9955,True,True


## 1. Export the discoveries as FASTA

All 412 go through rather than a top slice, so the rates in section 3.5 cover the whole set. Headers keep the id and consensus score so results map back.

In [2]:
fasta = f"screening/discovery_top{len(disc)}.fasta"
with open(fasta, "w") as fh:
    for _, r in disc.iterrows():
        fh.write(f">{r['peptide_id']}_c{r['consensus']}\n{r['sequence']}\n")

len(disc), fasta

(412, 'screening/discovery_top412.fasta')

## 2. Toxicity screening with ToxinPred2

Model 1 is the amino-acid-composition Random Forest. `-d 2` scores every peptide, not just the toxic ones, which is what the gate below needs.

Re-running overwrites `toxinpred2_discovery.csv`, which `phase5.5_tiered_table.py` also reads. Skip the cell if you just want to rebuild the shortlist.

In [3]:
!toxinpred2 -i {fasta} -o {TOXPRED} -m 1 -d 2

##############################################################################
# The program ToxinPred2.0 is developed for predicting Toxin and non toxin #
# protein from their primary sequence, developed by Prof G. P. S. Raghava's group. #
# ############################################################################
Summary of Parameters:
Input File:  screening/discovery_top412.fasta ; Model:  1 ; Threshold:  0.6
Output File:  screening/toxinpred2_discovery.csv ; Display:  2


## 3. Toxicity gate

A funnel, not a blended score. ToxinPred2 decides in or out, consensus decides the order. `tox_score` is kept because the calls sit close to the threshold.

In [4]:
tox = pd.read_csv(TOXPRED, keep_default_na=False)[["Sequence", "ML_Score"]]

m = disc.merge(tox, left_on="sequence", right_on="Sequence", how="left").drop(columns="Sequence")
assert m["ML_Score"].notna().all(), "some discoveries missing a ToxinPred2 score"

m["tox_score"] = m["ML_Score"] # P(toxin), so lower is safer
safe = m[m["ML_Score"] < TOX_THRESHOLD].sort_values("consensus", ascending=False, kind="stable").reset_index(drop=True)

cols = ["peptide_id", "sequence", "length", "sources", "consensus", "tox_score"]
safe[cols].to_csv(SHORTLIST, index=False)

# the same peptides as FASTA, which is what the AlgPred web form takes
with open(SHORTLIST_FASTA, "w") as fh:
    for _, r in safe.iterrows():
        fh.write(f">{r['peptide_id']}\n{r['sequence']}\n")

print(f"{len(m)} discoveries -> {len(safe)} non-toxic")
safe[cols].head(10)

412 discoveries -> 137 non-toxic


,peptide_id,sequence,length,sources,consensus,tox_score
0,CAND2716,FVAPFPEVF,9,bovine_alpha_s1_casein,0.9968,0.588
1,CAND3309,QPSSVDSQTAMVLVNAIVF,19,chicken_ovalbumin,0.9930,0.444
2,CAND3276,SPAQILQWQVLSNTVPAK,18,bovine_kappa_casein,0.9929,0.572
3,CAND1348,FVPAL,5,bovine_lactoferrin,0.9923,0.595
4,CAND2152,FVAPFPE,7,bovine_alpha_s1_casein,0.9919,0.595
5,CAND3491,HQGLPQEVLNENLLRFFVAPFPEVFGK,27,bovine_alpha_s1_casein,0.9899,0.593
6,CAND3348,ALPMHIRLSFNPTQLEEQCHI,21,bovine_beta_lactoglobulin,0.9884,0.515
7,CAND3554,TNDTPMIGTLAGANSLLNALPEEVIQHTFNLK,32,soybean_glycinin_g1,0.9884,0.404
8,CAND1400,IPAVF,5,bovine_beta_lactoglobulin,0.9883,0.576
9,CAND3289,EPMIGVNQELAYFYPELFR,19,bovine_alpha_s1_casein,0.9879,0.536


## 4. Allergenicity screening with AlgPred 2.0

**Manual step.** AlgPred 2.0 has no offline run in this project, so upload `screening/safe_shortlist.fasta` to the batch form and save the result as `screening/algpred2_safe.csv`.

https://webs.iiitd.edu.in/raghava/algpred2/batch_action.php

Settings used were the amino-acid-composition Random Forest at threshold 0.3. The download has columns `Subject`, `ML Score` and `Prediction`, with a space in the middle one.

Run the next cell once the file is in place.

In [5]:
assert os.path.exists(ALGPRED), f"{ALGPRED} not found, see the step above"

alg = pd.read_csv(ALGPRED, keep_default_na=False)
alg.columns = [c.strip() for c in alg.columns]
alg = alg.rename(columns={"Subject": "peptide_id", "ML Score": "alg_score",
                          "Prediction": "alg_pred"})
alg.head()

,peptide_id,alg_score,alg_pred
0,CAND2716,0.540,Allergen
1,CAND3309,0.340,Allergen
2,CAND3276,0.378,Allergen
3,CAND1348,0.411,Allergen
4,CAND2152,0.568,Allergen


## 5. Allergenicity gate and final leads

A lead survives only if it is both non-toxic and non-allergen. Both scores stay in the output because the calls sit close to their thresholds. Phase 5.5 re-partitions this set rather than trusting the gates.

In [6]:
final = safe.merge(alg[["peptide_id", "alg_score", "alg_pred"]], on="peptide_id", how="left")
assert final["alg_pred"].notna().all(), "some non-toxic leads missing an AlgPred call"

final = final[final["alg_pred"] == ALG_LABEL].sort_values("consensus", ascending=False, kind="stable").reset_index(drop=True)

final_cols = cols + ["alg_score"]
final[final_cols].to_csv(FINAL, index=False)

print(f"{len(safe)} non-toxic -> {len(final)} also non-allergen")
final[final_cols].head(10)

137 non-toxic -> 31 also non-allergen


,peptide_id,sequence,length,sources,consensus,tox_score,alg_score
0,CAND3348,ALPMHIRLSFNPTQLEEQCHI,21,bovine_beta_lactoglobulin,0.9884,0.515,0.276
1,CAND3554,TNDTPMIGTLAGANSLLNALPEEVIQHTFNLK,32,soybean_glycinin_g1,0.9884,0.404,0.195
2,CAND3323,ADAVTLDGGMVFEAGRDPYK,20,bovine_lactoferrin,0.9782,0.480,0.272
3,CAND3533,SAGWIIPMGILRPYLSWTESLEPLQGAVAK,30,bovine_lactoferrin,0.9776,0.508,0.210
4,CAND3584,TNDTPMIGTLAGANSLLNALPEEVIQHTFNLKSQQAR,37,soybean_glycinin_g1,0.9751,0.337,0.151
5,CAND3557,EIVPNSAEERLHSMKEGIHAQQKEPMIGVNQEL,33,bovine_alpha_s1_casein,0.9746,0.402,0.279
6,CAND3272,RVTEQESKPVQMMYQIGL,18,chicken_ovalbumin,0.9715,0.461,0.280
7,CAND3168,MAMGITDVFSSSANL,15,chicken_ovalbumin,0.9707,0.477,0.260
8,CAND2234,MMSFVSL,7,bovine_alpha_lactalbumin,0.9705,0.513,0.216
9,CAND3585,YDDFFLSSTQAQQSYLQGFSHNILETSFHSEFEEINR,37,soybean_beta_conglycinin,0.9564,0.533,0.291
